In [1]:
import optuna
import numpy as np
import pandas as pd

optuna.logging.set_verbosity(optuna.logging.WARNING)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error

import joblib
import os

import gc
gc.collect()

# Display Markdown
from IPython.display import display, Markdown

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

display(Markdown("### <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Libraries Loaded!</span>"))

C:\Users\roton\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Libraries Loaded!</span>

In [2]:
data_path    = "D:/Education/AiQuest/DS,ML,DL/Projects/House Price Prediction/data/USA_Housing.csv"
data= pd.read_csv(data_path)

display(Markdown("## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Data Loaded</span>"))

## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Data Loaded</span>

In [3]:
display(Markdown(f"**Rows:** {data.shape[0]}, **Columns:** {data.shape[1]}"))
display(data.head(3))

**Rows:** 5000, **Columns:** 7

,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population,Price,Address
0,79545.458574,5.682861,7.009188,4.09,23086.800503,1.059034e+06,"208 Michael Ferry Apt. 674\nLaurabury, NE 3701..."
1,79248.642455,6.002900,6.730821,3.09,40173.072174,1.505891e+06,"188 Johnson Views Suite 079\nLake Kathleen, CA..."
2,61287.067179,5.865890,8.512727,5.13,36882.159400,1.058988e+06,"9127 Elizabeth Stravenue\nDanieltown, WI 06482..."


# <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Load data and Split!</h2>

In [4]:
x_data = data.drop('Address', axis=1)

X = x_data.drop('Price', axis=1)
y = x_data['Price']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [6]:
display(Markdown(f"""
<span style='font-size:16px;'>
<b style='color:#27AE60; font-weight:bold; text-shadow:0 0 3px rgba(15,52,96,0.6);'>X_train :</b>
<span style='color:black; font-weight:bold;'>{X_train.shape}</span><br>
<b style='color:#27AE60; font-weight:bold; text-shadow:0 0 3px rgba(15,52,96,0.6);'>X_test  :</b>
<span style='color:black; font-weight:bold;'>{X_test.shape}</span><br>
<span style='font-size:16px;'>
<b style='color:#27AE60; font-weight:bold; text-shadow:0 0 3px rgba(15,52,96,0.6);'>y_train :</b>
<span style='color:black; font-weight:bold;'>{y_train.shape}</span><br>
<b style='color:#27AE60; font-weight:bold; text-shadow:0 0 3px rgba(15,52,96,0.6);'>y_test  :</b>
<span style='color:black; font-weight:bold;'>{y_test.shape}
</span>
"""))


<span style='font-size:16px;'>
<b style='color:#27AE60; font-weight:bold; text-shadow:0 0 3px rgba(15,52,96,0.6);'>X_train :</b>
<span style='color:black; font-weight:bold;'>(4000, 5)</span><br>
<b style='color:#27AE60; font-weight:bold; text-shadow:0 0 3px rgba(15,52,96,0.6);'>X_test  :</b>
<span style='color:black; font-weight:bold;'>(1000, 5)</span><br>
<span style='font-size:16px;'>
<b style='color:#27AE60; font-weight:bold; text-shadow:0 0 3px rgba(15,52,96,0.6);'>y_train :</b>
<span style='color:black; font-weight:bold;'>(4000,)</span><br>
<b style='color:#27AE60; font-weight:bold; text-shadow:0 0 3px rgba(15,52,96,0.6);'>y_test  :</b>
<span style='color:black; font-weight:bold;'>(1000,)
</span>


In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

display(Markdown("## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Features Scaled</span>"))

## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Features Scaled</span>

In [8]:
X_train_scaled

array([[-0.19049241, -0.12817719, -0.13160635,  0.12038585, -0.82761782],
       [-1.38876401,  0.43080443,  0.80028487, -0.55648895,  1.15829878],
       [-0.35012392,  0.46680752,  1.70375078,  0.03067955, -0.31904298],
       ...,
       [-0.22335061,  0.53809182, -0.36489661, -0.68697084,  0.11908894],
       [-0.92417067,  1.43077434,  2.26846315,  0.2753331 ,  1.39018355],
       [-0.69357335, -0.07762332,  0.89219611,  1.67801341, -0.00681852]],
      shape=(4000, 5))

 # <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Training Models!</h2>

In [9]:
models = {
    'Linear Regression': LinearRegression(),

    'Polynomial Degree 2': Pipeline([
        ('poly', PolynomialFeatures(degree=2)),
        ('linear', LinearRegression())
    ]),

    'Polynomial Degree 3': Pipeline([
        ('poly', PolynomialFeatures(degree=3)),
        ('linear', LinearRegression())
    ]),

    'Ridge α=0.1': Ridge(alpha=0.1),
    'Ridge α=1': Ridge(alpha=1),
    'Ridge α=10': Ridge(alpha=10),
    'Ridge α=100': Ridge(alpha=100),

    'Lasso α=0.01': Lasso(alpha=0.01),
    'Lasso α=0.1': Lasso(alpha=0.1),
    'Lasso α=1': Lasso(alpha=1),

    'KNN k=3': KNeighborsRegressor(n_neighbors=3),
    'KNN k=5': KNeighborsRegressor(n_neighbors=5),
    'KNN k=7': KNeighborsRegressor(n_neighbors=7)
}

In [10]:
results = []

for name, model in models.items():

    # Train
    model.fit(X_train_scaled, y_train)

    # Predictions
    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)

    # Metrics
    train_r2 = r2_score(y_train, train_pred)

    test_r2 = r2_score(y_test, test_pred)

    test_mse = mean_squared_error(
        y_test,
        test_pred
    )

    results.append(
        [
            name,
            train_r2,
            test_r2,
            test_mse
        ]
    )

display(Markdown("## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Models are Trained</span>"))

## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Models are Trained</span>

In [11]:
results

[['Linear Regression',
  0.9179787435623722,
  0.9179971706834331,
  10089009300.89399],
 ['Polynomial Degree 2',
  0.9181470377725716,
  0.9179137874167399,
  10099268148.792513],
 ['Polynomial Degree 3',
  0.9189091582539378,
  0.917397016631922,
  10162847726.447464],
 ['Ridge α=0.1', 0.9179787429296842, 0.9179971814072102, 10089007981.521389],
 ['Ridge α=1', 0.9179786803256885, 0.9179972203628644, 10089003188.711676],
 ['Ridge α=10', 0.9179724518665051, 0.9179919439085978, 10089652363.826902],
 ['Ridge α=100', 0.9173801693353605, 0.9174033945344908, 10162063037.444979],
 ['Lasso α=0.01', 0.9179787435623691, 0.9179971715138571, 10089009198.7249],
 ['Lasso α=0.1', 0.9179787435620383, 0.9179971842462386, 10089007632.228765],
 ['Lasso α=1', 0.9179787435312492, 0.9179972521079874, 10088999283.031284],
 ['KNN k=3', 0.9280263475385062, 0.8506921999534975, 18369704995.786713],
 ['KNN k=5', 0.9108554435118135, 0.8693170681585094, 16078241760.745293],
 ['KNN k=7', 0.9050246065793415, 0.87049

 # <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Comparison Table</h2>

In [12]:
comparison_table = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Train R²",
        "Test R²",
        "Test MSE"
    ]
)

comparison_table["Train R²"] = comparison_table["Train R²"].round(10)
comparison_table["Test R²"] = comparison_table["Test R²"].round(10)
comparison_table["Test MSE"] = comparison_table["Test MSE"].round(10)

display(Markdown("## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Comparision Table Created</span>"))

## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Comparision Table Created</span>

In [13]:
print("\nModel Comparison Table")
print("=" * 80)

print(comparison_table)


Model Comparison Table
                  Model  Train R²   Test R²      Test MSE
0     Linear Regression  0.917979  0.917997  1.008901e+10
1   Polynomial Degree 2  0.918147  0.917914  1.009927e+10
2   Polynomial Degree 3  0.918909  0.917397  1.016285e+10
3           Ridge α=0.1  0.917979  0.917997  1.008901e+10
4             Ridge α=1  0.917979  0.917997  1.008900e+10
5            Ridge α=10  0.917972  0.917992  1.008965e+10
6           Ridge α=100  0.917380  0.917403  1.016206e+10
7          Lasso α=0.01  0.917979  0.917997  1.008901e+10
8           Lasso α=0.1  0.917979  0.917997  1.008901e+10
9             Lasso α=1  0.917979  0.917997  1.008900e+10
10              KNN k=3  0.928026  0.850692  1.836970e+10
11              KNN k=5  0.910855  0.869317  1.607824e+10
12              KNN k=7  0.905025  0.870490  1.593387e+10


In [14]:
pd.set_option('display.float_format', lambda x: f'{x:.12f}')

sorted_comparison_table = comparison_table.sort_values(
    by='Test R²',
    ascending=False
).reset_index(drop=True)

display(Markdown("## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Comparision Table Sorted on Test R² </span>"))

## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Comparision Table Sorted on Test R² </span>

In [15]:

print("\nSorted Model Comparison Table")
print("=" * 80)

print(sorted_comparison_table)


Sorted Model Comparison Table
                  Model       Train R²        Test R²                 Test MSE
0             Lasso α=1 0.917978743500 0.917997252100 10088999283.031284332275
1             Ridge α=1 0.917978680300 0.917997220400 10089003188.711675643921
2           Lasso α=0.1 0.917978743600 0.917997184200 10089007632.228765487671
3           Ridge α=0.1 0.917978742900 0.917997181400 10089007981.521389007568
4          Lasso α=0.01 0.917978743600 0.917997171500 10089009198.724899291992
5     Linear Regression 0.917978743600 0.917997170700 10089009300.893989562988
6            Ridge α=10 0.917972451900 0.917991943900 10089652363.826902389526
7   Polynomial Degree 2 0.918147037800 0.917913787400 10099268148.792512893677
8           Ridge α=100 0.917380169300 0.917403394500 10162063037.444978713989
9   Polynomial Degree 3 0.918909158300 0.917397016600 10162847726.447463989258
10              KNN k=7 0.905024606600 0.870490490900 15933872685.276611328125
11              KNN k

# <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Best Model</h2>

In [16]:
best_model = sorted_comparison_table.iloc[0]

print("\nBest Model")
print("=" * 80)

print(f"Model    : {best_model['Model']}")
print(f"Test R²  : {best_model['Test R²']}")
print(f"Test MSE : {best_model['Test MSE']:.2f}")


Best Model
Model    : Lasso α=1
Test R²  : 0.9179972521
Test MSE : 10088999283.03


# <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Save Best Model Pipeline</h2>

In [17]:
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=1))
])

In [18]:
final_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True


In [19]:
model_path = r'D:\Education\AiQuest\DS,ML,DL\Projects\House Price Prediction\models'

os.makedirs(model_path, exist_ok=True)

joblib.dump(
    final_pipeline,
    os.path.join(model_path, 'best_model.pkl')
)
display(Markdown("## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Model saved successfully </span>"))

## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Model saved successfully </span>

# <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Key Findings!</h2>

1. Based on the Test R² score, Lasso Regression (α=1) achieved the highest performance<br>
<span style='font-weight:bold;font-size:14px; rgba(60, 117, 187, 0.6);'>Interpretation:</span>
- Excellent generalization.
- No overfitting.
- The relationship between features and price is largely linear.<br>
<span style='color:#27AE60;font-weight:bold;font-size:18px; rgba(15,52,96,0.6);'>This matches your EDA findings.:</span>
2. Polynomial Regression Does Not Help (mild overfitting when degree increase)
- Scatter plots indicate that most features have approximately linear positive relationships with house price.
- No complex non-linear patterns are evident.
3. Ridge and Lasso Behave Like Linear Regression<br>
<span style='font-weight:bold;font-size:14px; rgba(35, 95, 169, 0.6);'>This usually means:</span>
- Features are already well behaved.
- Multicollinearity is low.
- Regularization is not needed.
<br>
<span style='color:#27AE60;font-weight:bold;font-size:18px; rgba(15,52,96,0.6);'>Your correlation heatmap already suggested this.</span>

4. Large Ridge Penalty Hurts Performance<br>
<span style='font-weight:bold;font-size:14px; rgba(37, 100, 178, 0.6);'>When α becomes too large:</span>
- Coefficients shrink excessively.
- Underfitting begins.
5. KNN Performs Worse